# Example Usage of pairadigm Package
This notebook documents an example usage of the pairadigm package. It shows the step-by-step of using multiple LLM providers to generate and analyze LLM annotations of a construct and comparing those to simulated human annotations (see simulation annotation data.ipynb). Thanks for exploring! 

## Setup

In [4]:
import pandas as pd
import pairadigm as pdgm
import dotenv
import os

In [ ]:
# NOTE: To re-run this example, you will need to replace this path with one to your own API keys. The APIs used here were OPENAI and ANTHROPIC. Without those, you can at least see the example scoring, exploration, and evaluation below where I load the pickled pairaidgm of this example. 
 
dotenv.load_dotenv(dotenv_path='YOUR PATH HERE')

In [6]:
# Load example data. Here we are using EMOBANK data.
data = pd.read_csv("data/emobank_small_sample.csv")
data

,id,split,V,A,D,text
0,SemEval_1155,train,3.36,3.00,3.27,Microsoft to release next generation phone
1,The_Black_Willow_11596_11663,train,2.70,3.00,2.80,"Allan crouched over his desk once more, pen in..."
2,113CWL018_952_1022,train,3.90,3.10,3.40,Your contribution last year of helped us get w...
3,detroit_12208_12263,train,3.10,3.00,3.10,"Did we build sane, sustainable, survival shelt..."
4,SemEval_763,train,2.86,3.00,2.86,Secret hotels in Irish countryside
5,SemEval_597,train,3.12,3.12,3.12,Stenson defends his title at Dubai
6,Anti-Terrorist_4031_4279,test,2.92,3.00,2.91,"Beyond these two sources, there is virtually n..."
7,116CUL034_796_978,train,3.20,3.00,3.50,We have met with a number of successes along t...
8,Ant_Robot_19422_19434,test,3.00,2.78,3.00,The Behavior
9,Nathans_Bylichka_15318_15334,train,3.56,3.00,3.22,"“Thanks, Dvorov."


In [7]:
# Load our CGCOT Prompts
with open('data/cgcot_prompts/valence.txt', 'r') as file:
    val_prompts = file.read().splitlines()

In [8]:
# Initialize Pairadigm
p = pdgm.Pairadigm(
    data = data,
    item_id_name = 'id',
    text_name = 'text',
    cgcot_prompts = val_prompts,
    model_name = ['gpt-5-nano', 'claude-haiku-4-5'], 
    api_key = [os.getenv('OPENAI_API_KEY'), os.getenv('ANTHROPIC_API_KEY')],
    target_concept = 'valence'
)

In [9]:
# Review Clients
p.get_clients_info()

,index,model_name,provider
0,0,gpt-5-nano,openai
1,1,claude-haiku-4-5,anthropic


## Generating Pairwise Annotations

NOTE: Do not run these cells if you do not have API keys set up for the LLMs specified in the annotators list. 

In [10]:
# Get CGCOT Breakdowns
p.generate_breakdowns()

Breakdowns added to self.data with column name(s): CGCoT_Breakdown_gpt-5-nano, CGCoT_Breakdown_claude-haiku-4-5


In [11]:
# Pair items
p.generate_pairings(breakdowns=True) # breakdowns=True let's the object know it should attach the generated breakdowns, otherwise it will just generate the pairs.

Pairwise DataFrame with breakdowns created and stored in self.pairwise_df


,item1,item2,breakdown1_gpt-5-nano,breakdown2_gpt-5-nano,breakdown1_claude-haiku-4-5,breakdown2_claude-haiku-4-5
0,SemEval_1155,blog-new-year's-resolutions_1573_1737,Original Text: Microsoft to release next gener...,Original Text: An important premise to note he...,Original Text: Microsoft to release next gener...,Original Text: An important premise to note he...
1,blog-new-year's-resolutions_1573_1737,602CZL285_86_259,Original Text: An important premise to note he...,Original Text: Pat LaCrosse asked me to send t...,Original Text: An important premise to note he...,Original Text: Pat LaCrosse asked me to send t...
2,SemEval_763,Nathans_Bylichka_38817_38837,Original Text: Secret hotels in Irish countrys...,Original Text: “Didn’t I warn you?”\nPrompt 1 ...,Original Text: Secret hotels in Irish countrys...,Original Text: “Didn’t I warn you?”\nPrompt 1 ...
3,Nathans_Bylichka_45926_45965,hsus4_4933_5015,"Original Text: Isis collapsed on the fur, stri...",Original Text: REGISTRATION DOES NOT IMPLY END...,"Original Text: Isis collapsed on the fur, stri...",Original Text: REGISTRATION DOES NOT IMPLY END...
4,116CUL034_796_978,Ant_Robot_19422_19434,Original Text: We have met with a number of su...,Original Text: The Behavior\nPrompt 1 response...,Original Text: We have met with a number of su...,Original Text: The Behavior\nPrompt 1 response...
...,...,...,...,...,...,...
165,118CWL050_2436_2656,Nathans_Bylichka_12551_12560,Original Text: Will you make a financial gift ...,Original Text: What up?”\nPrompt 1 response: I...,Original Text: Will you make a financial gift ...,Original Text: What up?”\nPrompt 1 response: T...
166,SemEval_597,How_soon-Lebron-James_2193_2304,Original Text: Stenson defends his title at Du...,Original Text: I'm no longer the target demogr...,Original Text: Stenson defends his title at Du...,Original Text: I'm no longer the target demogr...
167,NYTnewswire7_3260_3393,118CWL050_2436_2656,"Original Text: ""We are also taking a real hard...",Original Text: Will you make a financial gift ...,"Original Text: ""We are also taking a real hard...",Original Text: Will you make a financial gift ...
168,wsj_2465_4206_4287,116CUL034_2074_2422,Original Text: Mr. Barco has refused U.S. troo...,"Original Text: Since its re-organization, MCCO...",Original Text: Mr. Barco has refused U.S. troo...,"Original Text: Since its re-organization, MCCO..."


In [12]:
# Generate LLM Pairwise Annotations
p.generate_pairwise_annotations()

[gpt-5-nano] Completed 50/170 comparisons
[gpt-5-nano] Completed 100/170 comparisons
[gpt-5-nano] Completed 150/170 comparisons
[claude-haiku-4-5] Completed 50/170 comparisons
[claude-haiku-4-5] Completed 100/170 comparisons
[claude-haiku-4-5] Completed 150/170 comparisons


,item1,item2,breakdown1_gpt-5-nano,breakdown2_gpt-5-nano,breakdown1_claude-haiku-4-5,breakdown2_claude-haiku-4-5,decision_gpt-5-nano,justification_gpt-5-nano,decision_claude-haiku-4-5,justification_claude-haiku-4-5
0,SemEval_1155,blog-new-year's-resolutions_1573_1737,Original Text: Microsoft to release next gener...,Original Text: An important premise to note he...,Original Text: Microsoft to release next gener...,Original Text: An important premise to note he...,Text2,FINAL ANSWER: Description 2\nJUSTIFICATION: De...,Text1,FINAL ANSWER: Description 1\n\nJUSTIFICATION: ...
1,blog-new-year's-resolutions_1573_1737,602CZL285_86_259,Original Text: An important premise to note he...,Original Text: Pat LaCrosse asked me to send t...,Original Text: An important premise to note he...,Original Text: Pat LaCrosse asked me to send t...,Text2,FINAL ANSWER: Description 2\nJUSTIFICATION: De...,Text1,FINAL ANSWER: Description 1\n\nJUSTIFICATION: ...
2,SemEval_763,Nathans_Bylichka_38817_38837,Original Text: Secret hotels in Irish countrys...,Original Text: “Didn’t I warn you?”\nPrompt 1 ...,Original Text: Secret hotels in Irish countrys...,Original Text: “Didn’t I warn you?”\nPrompt 1 ...,Text2,FINAL ANSWER: Description 2\nJUSTIFICATION: De...,Text1,FINAL ANSWER: Description 1\n\nJUSTIFICATION: ...
3,Nathans_Bylichka_45926_45965,hsus4_4933_5015,"Original Text: Isis collapsed on the fur, stri...",Original Text: REGISTRATION DOES NOT IMPLY END...,"Original Text: Isis collapsed on the fur, stri...",Original Text: REGISTRATION DOES NOT IMPLY END...,ERROR from pairadigm (not model): Regex match ...,,Text1,FINAL ANSWER: Description 1\n\nJUSTIFICATION: ...
4,116CUL034_796_978,Ant_Robot_19422_19434,Original Text: We have met with a number of su...,Original Text: The Behavior\nPrompt 1 response...,Original Text: We have met with a number of su...,Original Text: The Behavior\nPrompt 1 response...,Text1,FINAL ANSWER: Description 1\nJUSTIFICATION: De...,Text1,FINAL ANSWER: Description 1\n\nJUSTIFICATION: ...
...,...,...,...,...,...,...,...,...,...,...
165,118CWL050_2436_2656,Nathans_Bylichka_12551_12560,Original Text: Will you make a financial gift ...,Original Text: What up?”\nPrompt 1 response: I...,Original Text: Will you make a financial gift ...,Original Text: What up?”\nPrompt 1 response: T...,Text1,FINAL ANSWER: Description 1\nJUSTIFICATION: De...,Text1,FINAL ANSWER: Description 1\n\nJUSTIFICATION: ...
166,SemEval_597,How_soon-Lebron-James_2193_2304,Original Text: Stenson defends his title at Du...,Original Text: I'm no longer the target demogr...,Original Text: Stenson defends his title at Du...,Original Text: I'm no longer the target demogr...,Text2,FINAL ANSWER: Description 2\nJUSTIFICATION: De...,Text2,FINAL ANSWER: Description 2\n\nJUSTIFICATION: ...
167,NYTnewswire7_3260_3393,118CWL050_2436_2656,"Original Text: ""We are also taking a real hard...",Original Text: Will you make a financial gift ...,"Original Text: ""We are also taking a real hard...",Original Text: Will you make a financial gift ...,Text2,FINAL ANSWER: Description 2\nJUSTIFICATION: De...,Text2,FINAL ANSWER: Description 2\n\nJUSTIFICATION: ...
168,wsj_2465_4206_4287,116CUL034_2074_2422,Original Text: Mr. Barco has refused U.S. troo...,"Original Text: Since its re-organization, MCCO...",Original Text: Mr. Barco has refused U.S. troo...,"Original Text: Since its re-organization, MCCO...",Text2,FINAL ANSWER: Description 2\nJUSTIFICATION: De...,Text2,FINAL ANSWER: Description 2\n\nJUSTIFICATION: ...


A quick note here that you will notice in the decision column for gpt5-nano that there are ERRORs. Those stem from the model not following directions on formatting for pulling out its decision, even after a 2nd evaluation using the model again to try and detect the decision. 

## Scoring, Exploration, and Evaluation

In [ ]:
# Re-load picked version
p = pdgm.load_pairadigm(filepath='data/p_valence_example')

In [13]:
# Score items for both LLM decision columns
print("Scoring GPT-5-NANO....")
p.score_items(decision_col='decision_gpt-5-nano')
print("Scoring CLAUDE-HAIKI-4.5....")
p.score_items(decision_col='decision_claude-haiku-4-5')

Scoring GPT-5-NANO....
[gpt-5-nano] Bradley-Terry model fitted with 145 comparisons
[gpt-5-nano] Mean valence score: 0.570
[gpt-5-nano] Std valence score: 0.283
Score range: 0.000 to 1.000
25th percentile: 0.341
50th percentile (median): 0.592
75th percentile: 0.788

Highest scoring item on valence (score: 1.000):
We have met with a number of successes along the way, most notably the Summer Fun Line, the Metro Summer Bus Pass, and the development of ten neighborhood youth councils.

Lowest scoring item on valence (score: 0.000):
No one else said anything.
mean: 0.570
median: 0.592
std: 0.283
min: 0.000
max: 1.000
count: 30.000
Scoring CLAUDE-HAIKI-4.5....
[claude-haiku-4-5] Bradley-Terry model fitted with 170 comparisons
[claude-haiku-4-5] Mean valence score: 0.627
[claude-haiku-4-5] Std valence score: 0.274
Score range: 0.000 to 1.000
25th percentile: 0.376
50th percentile (median): 0.638
75th percentile: 0.855

Highest scoring item on valence (score: 1.000):
Will you make a financial

/Users/mlchrzan/Library/CloudStorage/OneDrive-Personal/Professional/Data Science/Personal Projects/pairadigm/pairadigm.py:2600: UserWarning: Some rows filtered out due to not containing 'Text1' or 'Text2' in the decision_col. If scoring human annotations, please adjust those values accordingly.
  warnings.warn("Some rows filtered out due to not containing 'Text1' or 'Text2' in the decision_col. If scoring human annotations, please adjust those values accordingly.")


,id,split,V,A,D,text,CGCoT_Breakdown_gpt-5-nano,CGCoT_Breakdown_claude-haiku-4-5,Bradley_Terry_Score_gpt-5-nano,Bradley_Terry_Score_claude-haiku-4-5
0,SemEval_1155,train,3.36,3.00,3.27,Microsoft to release next generation phone,Original Text: Microsoft to release next gener...,Original Text: Microsoft to release next gener...,0.501818,0.798228
1,The_Black_Willow_11596_11663,train,2.70,3.00,2.80,"Allan crouched over his desk once more, pen in...",Original Text: Allan crouched over his desk on...,Original Text: Allan crouched over his desk on...,0.461706,0.775444
2,113CWL018_952_1022,train,3.90,3.10,3.40,Your contribution last year of helped us get w...,Original Text: Your contribution last year of ...,Original Text: Your contribution last year of ...,0.855180,0.964682
3,detroit_12208_12263,train,3.10,3.00,3.10,"Did we build sane, sustainable, survival shelt...","Original Text: Did we build sane, sustainable,...","Original Text: Did we build sane, sustainable,...",0.646025,0.714132
4,SemEval_763,train,2.86,3.00,2.86,Secret hotels in Irish countryside,Original Text: Secret hotels in Irish countrys...,Original Text: Secret hotels in Irish countrys...,0.508183,0.940724
5,SemEval_597,train,3.12,3.12,3.12,Stenson defends his title at Dubai,Original Text: Stenson defends his title at Du...,Original Text: Stenson defends his title at Du...,0.095893,0.535517
6,Anti-Terrorist_4031_4279,test,2.92,3.00,2.91,"Beyond these two sources, there is virtually n...","Original Text: Beyond these two sources, there...","Original Text: Beyond these two sources, there...",0.554346,0.578082
7,116CUL034_796_978,train,3.20,3.00,3.50,We have met with a number of successes along t...,Original Text: We have met with a number of su...,Original Text: We have met with a number of su...,1.000000,0.998991
8,Ant_Robot_19422_19434,test,3.00,2.78,3.00,The Behavior,Original Text: The Behavior\nPrompt 1 response...,Original Text: The Behavior\nPrompt 1 response...,0.327894,0.181318
9,Nathans_Bylichka_15318_15334,train,3.56,3.00,3.22,"“Thanks, Dvorov.","Original Text: “Thanks, Dvorov.\nPrompt 1 resp...","Original Text: “Thanks, Dvorov.\nPrompt 1 resp...",0.929741,0.973907


In [ ]:
# Evaluate Annotator Transitivity in the pairs
p.check_transitivity()

{'decision_gpt-5-nano': (0.9924812030075187, 1, 133),
 'decision_claude-haiku-4-5': (0.9704433497536946, 6, 203)}

In the last cell's output you can see the impact of GPT5-nano not following the formatting rules, generating 70 less triplets.

In [15]:
# Explore differences in LLM Scores
from scipy.stats import pearsonr, spearmanr
import plotly.graph_objects as go
import numpy as np

# Extract the two Bradley-Terry score columns
x = p.scored_df['Bradley_Terry_Score_gpt-5-nano']
y = p.scored_df['Bradley_Terry_Score_claude-haiku-4-5']

# Calculate correlation statistics
pearson_r, pearson_p = pearsonr(x, y)
spearman_r, spearman_p = spearmanr(x, y)

# Calculate perpendicular distance from y=x line
distances = np.abs(y - x) / np.sqrt(2)

# Get text values for hover
text_values = p.scored_df['text']

# Create the plot
min_val = min(x.min(), y.min())
max_val = max(x.max(), y.max())

fig = go.Figure()

# Add scatter plot with hover text
fig.add_trace(go.Scatter(
    x=x,
    y=y,
    mode='markers',
    marker=dict(
        color=distances,
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title='Distance from y=x'),
        opacity=0.6
    ),
    text=text_values,
    hovertemplate='<b>Text:</b> %{text}<br>' +
                  '<b>gpt-5-nano:</b> %{x:.3f}<br>' +
                  '<b>claude-haiku-4-5:</b> %{y:.3f}<br>' +
                  '<extra></extra>',
    name='Data points'
))

# Add y=x line
fig.add_trace(go.Scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode='lines',
    line=dict(color='black', dash='dash'),
    opacity=0.5,
    name='y=x',
    hoverinfo='skip'
))

# Add stats annotation
stats_text = f'Pearson r = {pearson_r:.3f} (p = {pearson_p:.3f})<br>Spearman ρ = {spearman_r:.3f} (p = {spearman_p:.3f})'

fig.add_annotation(
    text=stats_text,
    xref='paper', yref='paper',
    x=0.05, y=0.95,
    xanchor='left', yanchor='top',
    showarrow=False,
    bgcolor='wheat',
    opacity=0.8,
    borderpad=10
)

fig.update_layout(
    title='Correlation between Model Bradley-Terry Scores',
    xaxis_title='Bradley-Terry Score (gpt-5-nano)',
    yaxis_title='Bradley-Terry Score (claude-haiku-4-5)',
    width=800,
    height=600,
    hovermode='closest',
    showlegend=True
)

fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')

fig.show()

## Comparing to "Human" Annotations

In [16]:
human_annots = pd.read_csv("data/emobank_small_sample_simAnnotations.csv")
human_annots = human_annots[['item1_id', 'item2_id', 'valence_human_1', 'valence_human_2', 'valence_human_3']].copy()
human_annots

,item1_id,item2_id,valence_human_1,valence_human_2,valence_human_3
0,SemEval_763,hotel-california_33157_33182,Text1,Text1,Text1
1,SemEval_1155,blog-new-year's-resolutions_1573_1737,Text1,Text2,Text1
2,Nathans_Bylichka_45926_45965,How_soon-Lebron-James_2193_2304,Text2,Text2,Text2
3,Ant_Robot_19422_19434,Nathans_Bylichka_15318_15334,Text2,Text2,Text1
4,detroit_12208_12263,blog-new-year's-resolutions_1573_1737,Text2,Text1,Text2
...,...,...,...,...,...
172,113CWL018_952_1022,blog-new-year's-resolutions_5010_5074,Text1,Text1,Text1
173,116CUL034_796_978,Nathans_Bylichka_12551_12560,Text1,Text1,Text1
174,detroit_12208_12263,Nathans_Bylichka_29483_29506,Text1,Text2,Text1
175,Uprooted_Farming-on-Sand_3238_3353,SemEval_1154,Text1,Text1,Text2


In [17]:
# Appending human annotations if not initialized with them
p.append_human_annotations(
    human_annots,
    annotator_names=['human_1', 'human_2', 'human_3'],
    item1_col='item1_id',
    item2_col='item2_id', 
    decision_cols=['valence_human_1', 'valence_human_2', 'valence_human_3'],
    overwrite=True
)

Successfully uploaded annotations for 'human_1'
  Coverage: 91/170 pairs (53.5%)
Successfully uploaded annotations for 'human_2'
  Coverage: 91/170 pairs (53.5%)
Successfully uploaded annotations for 'human_3'
  Coverage: 91/170 pairs (53.5%)

Human-annotated status: True
Total annotators: 3


In [24]:
# Check the IRR
p.irr()


INTER-RATER RELIABILITY RESULTS

HUMAN ANNOTATORS:
  Method: Krippendorff
  Score: 0.891
  Interpretation: Almost Perfect
  Annotators: 3
  Items: 91

LLM ANNOTATORS:
  Method: Cohens Kappa
  Score: 0.708
  Interpretation: Substantial
  Annotators: 2
  Items: 145

ALL ANNOTATORS:
  Method: Krippendorff
  Score: 0.891
  Interpretation: Almost Perfect
  Annotators: 5
  Items: 158



,group,error,method,score,n_annotators,n_items,interpretation
0,human,None,krippendorff,0.891414,3,91,Almost Perfect
1,llm,None,cohens_kappa,0.707914,2,145,Substantial
2,all,None,krippendorff,0.891149,5,158,Almost Perfect


In [18]:
# Check each LLM for passing the alt-test
p.alt_test(test_all_llms=True)

Testing all 2 LLM decision columns: ['decision_gpt-5-nano', 'decision_claude-haiku-4-5']
Using LLM decision column: decision_gpt-5-nano
Using LLM decision column: decision_gpt-5-nano

ALT-TEST RESULTS - GPT-5-NANO
Winning Rate (ω): 0.000
Advantage Probability: 0.589
Tested against 3 human annotators

Using LLM decision column: decision_claude-haiku-4-5

ALT-TEST RESULTS - CLAUDE-HAIKU-4-5
Winning Rate (ω): 0.000
Advantage Probability: 0.600
Tested against 3 human annotators



{'gpt-5-nano': (0.0, 0.5888888888888889), 'claude-haiku-4-5': (0.0, 0.6)}

In [21]:
# Check the DS Rank of each annotator
p.dawid_skene_annotator_ranking(random_seed=1234) # NOTE: random_seed must be set and I recommend testing multiple and examining the stability of the rankings to avoid seed hacking. This will be future functionality of the package. 

Ranking 5 annotators across 170 instances...

DAWID-SKENE ANNOTATOR RANKING
Converged at iteration: 100

Top 5 Most Reliable Annotators:
   rank                  annotator  reliability   type
0     1                    human_3     0.392334  Human
1     2                    human_2     0.344761  Human
2     3                    human_1     0.281622  Human
3     4  decision_claude-haiku-4-5     0.142986    LLM
4     5        decision_gpt-5-nano     0.030039    LLM




,annotator,reliability,type,rank
0,human_3,0.392334,Human,1
1,human_2,0.344761,Human,2
2,human_1,0.281622,Human,3
3,decision_claude-haiku-4-5,0.142986,LLM,4
4,decision_gpt-5-nano,0.030039,LLM,5


In [23]:
# Experimental combination of the methodologies
p.dawid_skene_alt_test(test_all_llms=True, max_iter=1000);

Testing all 2 LLM decision columns: ['decision_gpt-5-nano', 'decision_claude-haiku-4-5']

DAWID-SKENE VALIDATION RESULTS - GPT-5-NANO
Converged at iteration: 85

Annotator Reliability Weights:
  human_1: 0.1682
  human_2: 0.1267
  human_3: 0.1713

Advantage Probabilities per Annotator:

human_1:
  Mean margin: -0.2874
  p-value: 1.0000
  Corrected p-value: 1.0000
  Reject null: False

human_2:
  Mean margin: -0.2589
  p-value: 1.0000
  Corrected p-value: 1.0000
  Reject null: False

human_3:
  Mean margin: -0.2387
  p-value: 1.0000
  Corrected p-value: 1.0000
  Reject null: False

Overall Winning Rate (ω): 0.00


DAWID-SKENE VALIDATION RESULTS - CLAUDE-HAIKU-4-5
Converged at iteration: 86

Annotator Reliability Weights:
  human_1: 0.8318
  human_2: 0.8733
  human_3: 0.8287

Advantage Probabilities per Annotator:

human_1:
  Mean margin: -0.2534
  p-value: 1.0000
  Corrected p-value: 1.0000
  Reject null: False

human_2:
  Mean margin: -0.2294
  p-value: 1.0000
  Corrected p-value: 1.00

# Saving

If you were able to re-run this notebook using your own API keys and the pairadigm package, you can save your results using:

```python
p.save("your_desired_filename.pkl")
```

In [25]:
p.save("data/p_valence_example")

Pairadigm object saved successfully to: data/p_valence_example.pkl
